# 06 — Adaptive RAG

## Goal
This notebook builds a practical Adaptive RAG pipeline over the CIS Microsoft Windows 11 Enterprise Benchmark. It changes retrieval behavior based on the apparent complexity of the question:

- **Focused question:** ordinary vector similarity retrieval.
- **Broad or multi-part question:** generate focused subqueries, retrieve for each, then deduplicate evidence.
- **Weak-looking evidence:** make one broader fallback retrieval attempt.

This is a transparent educational implementation, not a production-grade classifier or confidence estimator. Benchmark recommendations alone do not establish whether a real Windows machine is compliant.

**Prerequisite:** Run `00b_ingestion_vector_store.ipynb` first.

## Architecture

```mermaid
flowchart TD
    A["User question"] --> B["Query analyzer"]
    B --> C{"Question type?"}
    C -->|"Focused"| D["Vector retrieval"]
    C -->|"Broad / multi-part"| E["Create focused subqueries"]
    E --> F["Retrieve for each subquery"]
    D --> G["Assess evidence coverage"]
    F --> G
    G --> H{"Enough evidence?"}
    H -->|"No"| I["Fallback broad retrieval"]
    H -->|"Yes"| J["Build bounded context"]
    I --> J
    J --> K["LLM generates grounded answer"]
    K --> L["Answer + source references"]

    classDef input fill:#dbeafe,stroke:#2563eb,color:#172554,stroke-width:2px;
    classDef route fill:#fef3c7,stroke:#d97706,color:#78350f,stroke-width:2px;
    classDef retrieve fill:#dcfce7,stroke:#16a34a,color:#14532d,stroke-width:2px;
    classDef decision fill:#fce7f3,stroke:#db2777,color:#831843,stroke-width:2px;
    classDef generate fill:#ede9fe,stroke:#7c3aed,color:#3b0764,stroke-width:2px;
    class A,L input;
    class B,C,E route;
    class D,F,I retrieve;
    class G,H decision;
    class J,K generate;
```

The Mermaid source is embedded in this notebook; display depends on the notebook renderer.

## 1. Imports and paths

In [ ]:
from pathlib import Path
import re
from collections import OrderedDict

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "vector_store").exists() and (PROJECT_ROOT.parent / "vector_store").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

VECTOR_STORE_DIR = PROJECT_ROOT / "vector_store" / "chroma_cis_windows11"
COLLECTION_NAME = "cis_windows11_benchmark"

print("Project root:", PROJECT_ROOT)
print("Vector store exists:", VECTOR_STORE_DIR.exists())

## 2. Select a model provider

Choose `ollama` or `groq`. For Groq, the key is read from `secrets/groq_api_key.txt` under the project root.

In [ ]:
PROVIDER = "ollama"  # "ollama" or "groq"
OLLAMA_MODEL = "gemma4:e2b"
GROQ_MODEL = "llama-3.3-70b-versatile"

if PROVIDER == "ollama":
    from langchain_ollama import ChatOllama
    llm = ChatOllama(model=OLLAMA_MODEL, temperature=0)
    print(f"Using Ollama model: {OLLAMA_MODEL}")
elif PROVIDER == "groq":
    from langchain_groq import ChatGroq
    key_path = PROJECT_ROOT / "secrets" / "groq_api_key.txt"
    if not key_path.exists():
        raise FileNotFoundError(
            f"Groq key file not found: {key_path}. Create it and put only your API key in it."
        )
    groq_api_key = key_path.read_text(encoding="utf-8").strip()
    if not groq_api_key:
        raise ValueError(f"Groq key file is empty: {key_path}")
    llm = ChatGroq(model=GROQ_MODEL, api_key=groq_api_key, temperature=0)
    print(f"Using Groq model: {GROQ_MODEL}")
else:
    raise ValueError("PROVIDER must be 'ollama' or 'groq'")

## 3. Open the persisted Chroma store

Use the same embedding model as the ingestion notebook.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

if not VECTOR_STORE_DIR.exists():
    raise FileNotFoundError(
        f"Vector store not found at {VECTOR_STORE_DIR}. "
        "Run notebooks/00b_ingestion_vector_store.ipynb first."
    )

vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    persist_directory=str(VECTOR_STORE_DIR),
    embedding_function=embedding_model,
)
print("Connected to:", COLLECTION_NAME)
print("Stored record count:", vector_store._collection.count())

## 4. Analyze query complexity

This simple, inspectable heuristic routes likely focused questions to one search and broad/multi-part questions to multi-query retrieval. The rules are intentionally visible so you can experiment with them.

In [ ]:
COMPLEXITY_MARKERS = (
    "compare", "comparison", "difference", "differences", "all ", "every ",
    "across", "multiple", "summarize", "summary", " versus ", " vs ",
    "steps", "process", "both", "several", "comprehensive"
)

def analyze_query(question: str) -> dict:
    q = question.strip().lower()
    words = re.findall(r"\b\w+\b", q)
    marker_hits = [marker for marker in COMPLEXITY_MARKERS if marker in q]
    is_complex = (
        len(words) >= 18 or len(marker_hits) >= 2 or
        any(marker in q for marker in ("compare", "differences", "versus", " vs "))
    )
    return {
        "word_count": len(words),
        "marker_hits": marker_hits,
        "route": "multi_query" if is_complex else "single_query",
    }

for question in [
    "What is the recommended screen lock setting?",
    "Compare password and account lockout recommendations, and explain how to verify them.",
]:
    print(analyze_query(question), "=>", question)

## 5. Retrieval helpers

For broad questions, the model creates a few focused search queries. The original question is always included, and a fallback keeps the pipeline usable if query generation fails.

In [ ]:
def doc_identity(doc):
    meta = doc.metadata or {}
    return (
        str(meta.get("source", "unknown")),
        meta.get("page"),
        meta.get("start_index"),
        doc.page_content[:180],
    )

def deduplicate_documents(documents):
    unique = OrderedDict()
    for doc in documents:
        unique.setdefault(doc_identity(doc), doc)
    return list(unique.values())

def make_subqueries(question: str, max_queries: int = 3) -> list[str]:
    from langchain_core.prompts import ChatPromptTemplate
    prompt = ChatPromptTemplate.from_messages([
        ("system",
         "Create up to {max_queries} short, distinct search queries for retrieving "
         "evidence from a Windows security benchmark. Return only one query per line, "
         "without numbering or commentary."),
        ("human", "Question: {question}")
    ])
    try:
        response = (prompt | llm).invoke({"question": question, "max_queries": max_queries})
        raw = response.content if hasattr(response, "content") else str(response)
        lines = [
            re.sub(r"^\s*(?:[-*]|\d+[.)])\s*", "", line).strip().strip('"')
            for line in raw.splitlines()
        ]
        lines = [line for line in lines if len(line) > 3]
        return list(dict.fromkeys([question] + lines[:max_queries - 1]))
    except Exception as exc:
        print("Subquery generation failed; using original question:", exc)
        return [question]

def retrieve_documents(question: str, k: int = 5) -> dict:
    analysis = analyze_query(question)
    if analysis["route"] == "single_query":
        queries = [question]
    else:
        queries = make_subqueries(question)

    docs = []
    for query in queries:
        docs.extend(vector_store.similarity_search(query, k=k))
    return {
        "analysis": analysis,
        "queries_used": queries,
        "documents": deduplicate_documents(docs)[:max(k, 8)],
    }

## 6. Evidence check and fallback

This check measures only the amount of retrieved text and number of chunks. It is **not** a semantic relevance score and cannot guarantee that the evidence answers the question.

In [ ]:
def evidence_looks_sufficient(documents, min_chars: int = 350) -> bool:
    if not documents:
        return False
    total_chars = sum(len(doc.page_content.strip()) for doc in documents)
    return len(documents) >= 2 and total_chars >= min_chars

def adaptive_retrieve(question: str, k: int = 5) -> dict:
    result = retrieve_documents(question, k=k)
    result["fallback_used"] = False

    if not evidence_looks_sufficient(result["documents"]):
        fallback_query = (
            "Windows 11 Enterprise security benchmark recommendations and "
            "configuration requirements relevant to: " + question
        )
        fallback_docs = vector_store.similarity_search(fallback_query, k=k)
        result["documents"] = deduplicate_documents(
            result["documents"] + fallback_docs
        )[:max(k, 8)]
        result["queries_used"].append(fallback_query)
        result["fallback_used"] = True

    result["evidence_sufficient_after_fallback"] = evidence_looks_sufficient(
        result["documents"]
    )
    return result

## 7. Grounded answer generation

The answer prompt asks the model to cite source labels and admit when the retrieved benchmark context does not support a claim.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

def source_label(doc, index: int) -> str:
    meta = doc.metadata or {}
    source = Path(str(meta.get("source", "unknown source"))).name
    page = meta.get("page", "unknown")
    start = meta.get("start_index")
    label = f"[S{index}] {source}, page {page}"
    if start is not None:
        label += f", chunk start {start}"
    return label

def build_context(documents, max_chars_per_doc: int = 1800) -> str:
    parts = []
    for i, doc in enumerate(documents, start=1):
        parts.append(f"{source_label(doc, i)}\n{doc.page_content.strip()[:max_chars_per_doc]}")
    return "\n\n---\n\n".join(parts)

answer_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You explain a Windows 11 security benchmark. Answer only from the supplied context. "
     "Cite claims using source labels such as [S1]. If context is insufficient, say what is "
     "missing. Do not claim that a real machine is compliant based only on benchmark text. "
     "Distinguish recommendations from evidence of the machine's current state."),
    ("human", "Question: {question}\n\nRetrieved context:\n{context}")
])

def adaptive_rag(question: str, k: int = 5) -> dict:
    result = adaptive_retrieve(question, k=k)
    context = build_context(result["documents"])
    response = (answer_prompt | llm).invoke({
        "question": question,
        "context": context or "[No usable context retrieved.]",
    })
    answer = response.content if hasattr(response, "content") else str(response)
    return {**result, "context": context, "answer": answer}

## 8. Run a question

In [ ]:
question = "What does the benchmark recommend about screen locking?"
result = adaptive_rag(question, k=5)

print("Route:", result["analysis"]["route"])
print("Queries used:")
for query in result["queries_used"]:
    print(" -", query)
print("Fallback used:", result["fallback_used"])
print("\nANSWER\n")
print(result["answer"])

## 9. Inspect the retrieved evidence

In [ ]:
for i, doc in enumerate(result["documents"], start=1):
    print("=" * 88)
    print(source_label(doc, i))
    print(doc.page_content[:1000])
    print()

## 10. Compare routes

Run a focused question and a multi-part question to observe how the selected retrieval strategy changes.

In [ ]:
test_questions = [
    "What is the recommended screen lock setting?",
    "Compare account lockout and password recommendations, and explain how an administrator could verify the configured settings.",
]
for q in test_questions:
    r = adaptive_retrieve(q, k=4)
    print("\nQUESTION:", q)
    print("ROUTE:", r["analysis"]["route"])
    print("QUERY COUNT:", len(r["queries_used"]))
    print("DOCUMENT COUNT:", len(r["documents"]))
    print("FALLBACK USED:", r["fallback_used"])
    print("EVIDENCE SUFFICIENT:", r["evidence_sufficient_after_fallback"])
    print("QUERIES:", r["queries_used"])

## 11. Lightweight checks

In [ ]:
assert analyze_query("What is the recommended screen lock setting?")["route"] == "single_query"
assert analyze_query("Compare password and account lockout recommendations, and explain the differences.")["route"] == "multi_query"
assert deduplicate_documents([]) == []
assert evidence_looks_sufficient([], min_chars=1) is False
print("Basic adaptive routing/helper checks passed.")

## Summary and limitations

This notebook demonstrates adaptive routing, multi-query retrieval, fallback retrieval, source inspection, and grounded generation.

Limitations:
1. The query analyzer is a heuristic and may misroute questions.
2. The evidence check measures quantity, not semantic relevance or coverage.
3. Multi-query retrieval increases latency and may add redundant chunks.
4. Retrieved text and citations must be inspected; a plausible answer is not proof of support.
5. Evaluate on representative questions with expected source passages before relying on the pipeline.
6. The CIS benchmark states recommendations. Assessing actual compliance requires trusted machine evidence too.